# 00 · Build data

Prose corpus → non-IID clients → 4 task datasets → splits, on `MyDrive/FedDAPT`.

**Full real build:** run every cell (AIT logs + local teacher). **Plumbing check:** skip Step A & B; leave `teacher=None` (`explain_log`/`verdict` come out empty — expected).

In [ ]:
import os, sys
!test -d /content/fedapt || git clone https://github.com/dsuyu1/fedapt.git /content/fedapt
os.chdir('/content/fedapt')
!pip install -e "."
sys.path.insert(0, '/content/fedapt/src')
import fedapt; print('fedapt OK:', fedapt.__file__)
from google.colab import drive; drive.mount('/content/drive')

Cloning into '/content/fedapt'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 130 (delta 45), reused 120 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 301.13 KiB | 3.86 MiB/s, done.
Resolving deltas: 100% (45/45), done.
Obtaining file:///content/fedapt
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fedapt (pyproject.toml) ... done
  Created wheel for fedapt: filename=fedapt-0.1.0-0.editable-py3-none-any.whl size=2336 sha256=0c6e565263388695707c5f3eeaba39828c5e124a6e69c5b78dedcf1363ab90f8
  Stored in directory: /tmp/pip-ephem-wheel-cache-0jr0i44e/wheels/bf/53/c3/4dc2e7b687f07244f6f1c8bad3be6806a5e7dc97dad923ea44
Successfully built fedapt


### Load config

In [ ]:
from fedapt.config import load_config
from fedapt import corpus, clients, tasks, splits
from fedapt.judge import make_llm
cfg = load_config()
teacher = None
print('root =', cfg.root)

## Step A — Log data (AIT): download → convert → point config
Both classes from one matched environment. Raw AIT (many GB) goes to ephemeral `/content`; we keep only the small converted `.jsonl` on **Drive**. Skip this section for a prose-only check.

In [ ]:
# 1) download ONE AIT testbed to ephemeral local disk (not Drive)
AIT_ZIP = 'russellmitchell.zip'   # smallest at https://zenodo.org/record/5789064 (fox/harrison/…)
get_ipython().system(f'wget -q --show-progress https://zenodo.org/record/5789064/files/{AIT_ZIP} -O /content/ait.zip')
get_ipython().system('cd /content && unzip -q -o ait.zip')
import glob
_g = glob.glob('/content/**/gather', recursive=True)
AIT_SRC = os.path.dirname(_g[0]) if _g else None
print('AIT source dir =', AIT_SRC)

In [ ]:
# 2) convert -> SMALL jsonl on DRIVE (persists). CHECK the printed class balance!
assert AIT_SRC, 'no gather/ dir under /content — did the download/unzip succeed?'
get_ipython().system('mkdir -p /content/drive/MyDrive/FedDAPT/normalized')
get_ipython().system(f'python scripts/convert_dataset.py --dataset ait --src {AIT_SRC} --out /content/drive/MyDrive/FedDAPT/normalized/ait.jsonl')
# the line 'class balance: X malicious / Y benign' MUST show X > 0 (else tell Claude).

In [ ]:
# 3) point the pipeline at the (Drive) logs and re-read config
os.environ['FEDDAPT_LOG_SOURCES'] = '/content/drive/MyDrive/FedDAPT/normalized'
cfg = load_config()
print('log sources:', cfg.log_source_paths)

## Step B — Local teacher (Ollama on Colab's GPU)
Real targets, free, no rate limits. Run the install cell, then set the teacher. Skip both to leave `teacher=None`.

In [ ]:
# run a LOCAL teacher/judge on Colab's GPU via Ollama (no API key, no rate limits)
# T4: qwen3:14b / gemma3:12b.  A100: qwen3:32b / gemma3:27b.
get_ipython().system('curl -fsSL https://ollama.com/install.sh | sh')
import subprocess, time
subprocess.Popen(['ollama', 'serve']); time.sleep(5)
get_ipython().system('ollama pull qwen3:14b')
os.environ['FEDDAPT_OLLAMA_HOST'] = 'http://localhost:11434'

In [ ]:
teacher = make_llm('ollama:qwen3:14b')   # or make_llm('claude-haiku-4-5') for API (needs .[eval] + key)
print('teacher set:', teacher is not None)

## Step C — Build the data layer
Reuses the prose corpus if already on Drive.

In [ ]:
if not os.path.exists(os.path.join(cfg.corpus_dir, 'prose_corpus.jsonl')):
    corpus.build_corpus(cfg)
else:
    print('prose corpus already built — skipping re-fetch')
clients.build_clients(cfg)
tasks.build_tasks(cfg, teacher=teacher)   # watch the '⚠ fell back' line — should be 0
splits.build_splits(cfg)

**Check:** `explain_log`/`verdict` now >0, verdict has both classes, fallbacks 0. Then → **01**.